# 🧱 Part 23: On-Policy Distillation

> **Previous context**: Standard distillation often trains on teacher or dataset distributions. OPD focuses on the student's own generated distribution.
> **Goal for this part**: Understand exposure bias, Forward/Reverse KL, k1/k2/k3 estimators, OPSD, and the paper taxonomy.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Exposure bias

A student may train on one distribution but generate from another. Errors compound when the model must continue from its own imperfect outputs.

## 1. On-policy idea

On-policy distillation samples from the student's current behavior and teaches on the states it actually visits.

## 2. KL directions

Forward KL and Reverse KL encourage different behavior: coverage versus mode-seeking. Estimators approximate these objectives from samples.

## 3. Taxonomy

The notebook organizes recent OPD-related papers by objective, sampling strategy, and correction method.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] OPD trains on the student's own generated states.
- [ ] KL direction changes what behavior is encouraged.
- [ ] Estimator choice matters for stability and bias.

Next, continue through the code cells for the Evaluation & Deployment part and inspect the printed observations.


In [1]:
import numpy as np

np.random.seed(42)

In [2]:
# Teaching note: follow this line to see the main step.
vocab = {'Read the values printed above and connect them to the concept in this cell.': 0, 'Read the values printed above and connect them to the concept in this cell.': 1, 'Read the values printed above and connect them to the concept in this cell.': 2, 'Read the values printed above and connect them to the concept in this cell.': 3, 'Read the values printed above and connect them to the concept in this cell.': 4, 'Read the values printed above and connect them to the concept in this cell.': 5, 'Read the values printed above and connect them to the concept in this cell.': 6, 'Read the values printed above and connect them to the concept in this cell.': 7, 'Read the values printed above and connect them to the concept in this cell.': 8, 'Read the values printed above and connect them to the concept in this cell.': 9}
id2token = {v: k for k, v in vocab.items()}
vocab_size = len(vocab)

def softmax(logits):
    logits = np.array(logits, dtype=np.float64)
    exp_logits = np.exp(logits - logits.max())
    return exp_logits / exp_logits.sum()

def log_softmax(logits):
    logits = np.array(logits, dtype=np.float64)
    m = logits.max()
    return logits - m - np.log(np.exp(logits - m).sum())

# Teaching note: follow this line to see the main step.
def student_model(input_ids):
    rng = np.random.RandomState(sum(input_ids) * 7 + 42)
    return rng.randn(vocab_size) * 2

# Teaching note: follow this line to see the main step.
def teacher_model(input_ids):
    rng = np.random.RandomState(sum(input_ids) * 13 + 7)
    return rng.randn(vocab_size) * 1.2

print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.

In [3]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
train_prefix = [0, 1, 2]  # Teaching note: follow this line to see the main step.
print(f"Read the values printed above and connect them to the concept in this cell.{[id2token[i] for i in train_prefix]}Read the values printed above and connect them to the concept in this cell.")
t_logits = teacher_model(train_prefix)
t_probs = softmax(t_logits)
top3 = np.argsort(t_probs)[-3:][::-1]
print('Read the values printed above and connect them to the concept in this cell.')
for idx in top3:
    idx = int(idx)
    print(f"    '{id2token[idx]}': {t_probs[idx]:.3f}")
print()

# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
np.random.seed(99)
student_sequence = [0]  # Teaching note: follow this line to see the main step.
for step in range(3):
    s_logits = student_model(student_sequence)
    s_probs = softmax(s_logits)
    next_token = np.random.choice(vocab_size, p=s_probs)
    student_sequence.append(next_token)
    print(f"  Step {step+1}: Input {[id2token[i] for i in student_sequence[:-1]]} "
          f"Read the values printed above and connect them to the concept in this cell.{id2token[next_token]}' (probability={s_probs[next_token]:.3f})")
print()

# Teaching note: follow this line to see the main step.
weird_prefix = student_sequence[:-1]
s_logits_weird = student_model(weird_prefix)
s_probs_weird = softmax(s_logits_weird)
entropy = -np.sum(s_probs_weird * np.log(s_probs_weird + 1e-10))
print(f"Read the values printed above and connect them to the concept in this cell.{[id2token[i] for i in weird_prefix]}")
print(f"Read the values printed above and connect them to the concept in this cell.{entropy:.3f}Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.{id2token[np.argmax(s_probs_weird)]}' ({np.max(s_probs_weird):.3f})")
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values print

In [4]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

prompt = [0, 1]  # Teaching note: follow this line to see the main step.
sequence = prompt.copy()
temperature = 1.0

print(f"Prompt: {[id2token[i] for i in prompt]}")
print()

# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
for step in range(3):
    logits = student_model(sequence)
    probs = softmax(logits / temperature)
    next_token = np.random.choice(vocab_size, p=probs)
    sequence.append(next_token)
    print(f"Read the values printed above and connect them to the concept in this cell.{[id2token[i] for i in sequence[:-1]]}Read the values printed above and connect them to the concept in this cell.{id2token[next_token]}")

print(f"Read the values printed above and connect them to the concept in this cell.{[id2token[i] for i in sequence]}")
print()

# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
for t in range(len(prompt), len(sequence)):
    prefix = sequence[:t]
    sampled_token = sequence[t]
    
    s_logits = student_model(prefix)
    t_logits = teacher_model(prefix)
    s_logp = log_softmax(s_logits)[sampled_token]
    t_logp = log_softmax(t_logits)[sampled_token]
    
    advantage = t_logp - s_logp
    
    print(f"Read the values printed above and connect them to the concept in this cell.{t}: prefix={[id2token[i] for i in prefix]}")
    print(f"Read the values printed above and connect them to the concept in this cell.{id2token[sampled_token]}Read the values printed above and connect them to the concept in this cell.{s_logp:.3f}, teacherlogp={t_logp:.3f}")
    print(f"    advantage = {advantage:+.3f} → {'Read the values printed above and connect them to the concept in this cell.' if advantage > 0 else 'Read the values printed above and connect them to the concept in this cell.'}")

print()
print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values print

In [5]:
# Teaching note: follow this line to see the main step.
print('"=== Forward KL vs Reverse KL Intuition ==="')
print()

# Teaching note: follow this line to see the main step.
teacher_probs = np.array([0.40, 0.35, 0.25])
styles = ['Read the values printed above and connect them to the concept in this cell.', 'Read the values printed above and connect them to the concept in this cell.', 'Read the values printed above and connect them to the concept in this cell.']

print('Read the values printed above and connect them to the concept in this cell.')
for i, (style, prob) in enumerate(zip(styles, teacher_probs)):
    bar = "█" * int(prob * 40)
    print(f"  {style}: {prob:.0%} {bar}")
print()

# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
student_avg_probs = np.array([0.33, 0.33, 0.34])  # Teaching note: follow this line to see the main step.
forward_kl = np.sum(teacher_probs * (np.log(teacher_probs + 1e-10) - np.log(student_avg_probs + 1e-10)))

print('Read the values printed above and connect them to the concept in this cell.')
for i, (style, prob) in enumerate(zip(styles, student_avg_probs)):
    bar = "█" * int(prob * 40)
    print(f"  {style}: {prob:.0%} {bar}")
print(f"  Forward KL = {forward_kl:.4f}")
print(f"Read the values printed above and connect them to the concept in this cell.")
print()

# Teaching note: follow this line to see the main step.
student_spec_probs = np.array([0.05, 0.85, 0.10])  # Teaching note: follow this line to see the main step.
reverse_kl = np.sum(student_spec_probs * (np.log(student_spec_probs + 1e-10) - np.log(teacher_probs + 1e-10)))

print('Read the values printed above and connect them to the concept in this cell.')
for i, (style, prob) in enumerate(zip(styles, student_spec_probs)):
    bar = "█" * int(prob * 40)
    print(f"  {style}: {prob:.0%} {bar}")
print(f"  Reverse KL = {reverse_kl:.4f}")
print(f"Read the values printed above and connect them to the concept in this cell.")
print()

print(f"Read the values printed above and connect them to the concept in this cell.{forward_kl:.4f}) vs Reverse KL ({reverse_kl:.4f})")
print('Read the values printed above and connect them to the concept in this cell.')

=== Forward KL vs Reverse KL Intuition ===
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.  Forward KL = 0.0207
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and

In [6]:
# Teaching note: follow this line to see the main step.
prefix_demo = [0, 1, 2]  # Teaching note: follow this line to see the main step.

print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
t_logits = teacher_model(prefix_demo)
t_logprobs = log_softmax(t_logits)
sampled_token = 3  # Teaching note: follow this line to see the main step.

print(f"1. sampled-token:")
print(f"Read the values printed above and connect them to the concept in this cell.{id2token[sampled_token]}Read the values printed above and connect them to the concept in this cell.{t_logprobs[sampled_token]:.3f}")
print(f"Read the values printed above and connect them to the concept in this cell.")
print()

# Teaching note: follow this line to see the main step.
k = 5
topk_idx = np.argsort(t_logprobs)[-k:][::-1]
print(f"2. top-{k}:")
for idx in topk_idx:
    print(f"Read the values printed above and connect them to the concept in this cell.{id2token[idx]}Read the values printed above and connect them to the concept in this cell.{t_logprobs[idx]:.3f}")
print()

# Full-vocab
print(f"3. full-vocab:")
print(f"Read the values printed above and connect them to the concept in this cell.{vocab_size}Read the values printed above and connect them to the concept in this cell.")
all_probs = softmax(t_logits)
for i, tok in id2token.items():
    bar = "█" * int(all_probs[i] * 40)
    print(f"   {tok:4s}: {all_probs[i]:.4f}  {bar}")

Read the values printed above and connect them to the concept in this cell.
1. sampled-token:Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
2. top-5:Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
3. full-vocab:Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the con

In [7]:
def k1_estimator(s_logp, t_logp):
    'Read the values printed above and connect them to the concept in this cell.'
    return s_logp - t_logp

def k2_estimator(s_logp, t_logp):
    'Read the values printed above and connect them to the concept in this cell.'
    diff = s_logp - t_logp
    return 0.5 * diff * diff

def k3_estimator(s_logp, t_logp):
    'Read the values printed above and connect them to the concept in this cell.'
    ratio = t_logp - s_logp  # log(P_T / P_S)
    r = np.exp(ratio)         # P_T / P_S
    return r - ratio - 1

# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print(f"{'Read the values printed above and connect them to the concept in this cell.':>10} {"'teacherlogp'":>10} {'k1':>10} {'k2':>10} {'k3':>10}  Note")
print("-" * 65)

cases = [
    (-0.5, -0.5, 'Read the values printed above and connect them to the concept in this cell.'),
    (-1.0, -0.5, 'Read the values printed above and connect them to the concept in this cell.'),
    (-0.5, -1.0, 'Read the values printed above and connect them to the concept in this cell.'),
    (-2.0, -0.5, 'Read the values printed above and connect them to the concept in this cell.'),
    (-0.5, -2.0, 'Read the values printed above and connect them to the concept in this cell.'),
    (-3.0, -0.1, 'Read the values printed above and connect them to the concept in this cell.'),
]

for s_lp, t_lp, desc in cases:
    k1 = k1_estimator(s_lp, t_lp)
    k2 = k2_estimator(s_lp, t_lp)
    k3 = k3_estimator(s_lp, t_lp)
    print(f"{s_lp:>+10.2f} {t_lp:>+10.2f} {k1:>+10.4f} {k2:>10.4f} {k3:>10.4f}  ← {desc}")

print()
print('Loss')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.-----------------------------------------------------------------
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
LossRead the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

In [8]:
def simulate_opd_training(prompt_ids, num_gen_tokens=3, mode="sampled_token_k3", topk=5):
    'Read the values printed above and connect them to the concept in this cell.'
    
    # Teaching note: follow this line to see the main step.
    sequence = list(prompt_ids).copy()
    temp = 1.0
    
    print('Read the values printed above and connect them to the concept in this cell.')
    print(f"  Prompt: {[id2token[i] for i in prompt_ids]}")
    for step in range(num_gen_tokens):
        logits = student_model(sequence)
        probs = softmax(logits / temp)
        next_token = np.random.choice(vocab_size, p=probs)
        sequence.append(next_token)
    
    generated = sequence[len(prompt_ids):]
    print(f"Read the values printed above and connect them to the concept in this cell.{[id2token[i] for i in generated]}")
    
    # Teaching note: follow this line to see the main step.
    print(f"Loss{mode})")
    total_loss = 0
    
    for t in range(len(prompt_ids), len(sequence)):
        prefix = sequence[:t]
        sampled_token = sequence[t]
        
        if mode == "sampled_token_k3":
            s_logp = log_softmax(student_model(prefix))[sampled_token]
            t_logp = log_softmax(teacher_model(prefix))[sampled_token]
            loss = k3_estimator(s_logp, t_logp)
            print(f"Read the values printed above and connect them to the concept in this cell.{t}: '{id2token[sampled_token]}' | s_logp={s_logp:.3f} t_logp={t_logp:.3f} | k3={loss:.4f}")
        
        elif mode == "topk_rkl":
            t_logp = log_softmax(teacher_model(prefix))
            topk_idx = np.argsort(t_logp)[-topk:][::-1]
            t_renorm = softmax(t_logp[topk_idx])
            s_logp = log_softmax(student_model(prefix))
            s_renorm = softmax(s_logp[topk_idx])
            loss = np.sum(s_renorm * (np.log(s_renorm + 1e-10) - np.log(t_renorm + 1e-10)))
            print(f"Read the values printed above and connect them to the concept in this cell.{t}: top-{topk}={[id2token[i] for i in topk_idx]} | RKL={loss:.4f}")
        
        total_loss += loss
    
    avg_loss = total_loss / len(generated)
    print(f"Loss{total_loss:.4f}Read the values printed above and connect them to the concept in this cell.{avg_loss:.4f}")
    print(f"Read the values printed above and connect them to the concept in this cell.")
    return avg_loss

np.random.seed(42)
print('Read the values printed above and connect them to the concept in this cell.')
simulate_opd_training([0, 1], num_gen_tokens=3, mode="sampled_token_k3")

### Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
LossRead the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
LossRead the values printed above and connect them to the concept in this cell.

np.float64(2.283848208244739)

In [9]:
np.random.seed(42)
print('Read the values printed above and connect them to the concept in this cell.')
simulate_opd_training([0, 1], num_gen_tokens=3, mode="topk_rkl", topk=5)

### Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
LossRead the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
LossRead the values printed above and connect them to the concept in this cell.

np.float64(0.7655468551277211)

---

1. .2. .3. .4. .5. .6. .7. .8. .9. .
10. .11. .12. .13. .14. .15. .
..
..
- awesome-on-policy-distillation: https://github.com/chrisliu298/awesome-on-policy-distillation- OPD Survey: https://arxiv.org/abs/2604.00626- MiniLLM: https://arxiv.org/abs/2306.08543- GKD: https://arxiv.org/abs/2306.13649- ExOPD: https://arxiv.org/abs/2602.12125- OPSD: https://arxiv.org/abs/2601.18734- Pitfalls (Zhu et al.): https://arxiv.org/abs/2605.11182- OGLS-SD: https://arxiv.org/abs/2605.12400- Thinking Machines blog: https://thinkingmachines.ai/blog/on-policy-distillation/- TRL DistillationTrainer: https://huggingface.co/docs/trl

---

1. .2. .3. .4. .5. .6. .7. .8. .9. .10. .
..